# Mapa afastado

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import contextily as ctx

# ==============================
# 1. Ler CSV
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1853350, 1856050
este_min, este_max = 2791350, 2794450

ortho = ortho[
    (ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
    (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)
]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Plot só o mapa (sem pontos)
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

# Definir limites com base na extensão dos pontos ORTHO
xmin, ymin, xmax, ymax = gdf_ortho.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Adicionar o mapa base
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Título e formatação
ax.set_title("Área da Barragem de Alqueva", fontsize=13)
ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import contextily as ctx

# ==============================
# 1. Ler CSV
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1853350, 1856050
este_min, este_max = 2791350, 2794450

ortho = ortho[
    (ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
    (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)
]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Plot apenas pontos ORTHO
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

gdf_ortho.plot(ax=ax, color='yellow', edgecolor='black', markersize=30, alpha=0.8, label='Pontos ORTHO')

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Localização dos pontos ORTHO (Área da Barragem de Alqueva)", fontsize=13)
ax.legend()
ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1853350, 1856050
este_min, este_max = 2791350, 2794450

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Determinar grelha contínua e centrada nos pontos ortho
# ==============================
grid_size = 25  # metros

# Determinar o canto inferior esquerdo da grelha
x_min = ortho['easting'].min()
x_max = ortho['easting'].max()
y_min = ortho['northing'].min()
y_max = ortho['northing'].max()

# ⚙️ Ajustar limites para que o centro das células bata nos pontos ortho
# Isto é: deslocar o início da grelha meio passo para trás
x_min_aligned = x_min - grid_size / 2
y_min_aligned = y_min - grid_size / 2

# Criar limites regulares e contínuos
x_edges = np.arange(x_min_aligned, x_max + grid_size, grid_size)
y_edges = np.arange(y_min_aligned, y_max + grid_size, grid_size)

# Criar células contínuas
grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Plot: Grelha contínua + pontos ortho
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

grid.boundary.plot(ax=ax, color='red', linewidth=0.3, alpha=0.8, label='Grelha (100m contínua)')
#gdf_ortho.plot(ax=ax, color='white', edgecolor='black', markersize=25, alpha=0.9, label='Pontos ORTHO (centros)')
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Grelha contínua centrada nos pontos ORTHO (100m)", fontsize=13)
ax.legend()
ax.set_axis_off()

plt.tight_layout()
plt.show()

# Mapa localizado

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSV
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

ortho = ortho[
    (ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
    (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)
]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Plot apenas o mapa base (sem grelha nem pontos)
# ==============================
fig, ax = plt.subplots(figsize=(8, 6))

# Definir limites com base na extensão dos pontos ORTHO
xmin, ymin, xmax, ymax = gdf_ortho.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Adicionar mapa base
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Título e formatação
#ax.set_title("Área selecionada sobre a Barragem de Alqueva", fontsize=13)
ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Determinar grelha contínua e centrada nos pontos ortho
# ==============================
grid_size = 25  # metros

# Determinar o canto inferior esquerdo da grelha
x_min = ortho['easting'].min()
x_max = ortho['easting'].max()
y_min = ortho['northing'].min()
y_max = ortho['northing'].max()

# ⚙️ Ajustar limites para que o centro das células bata nos pontos ortho
# Isto é: deslocar o início da grelha meio passo para trás
x_min_aligned = x_min - grid_size / 2
y_min_aligned = y_min - grid_size / 2

# Criar limites regulares e contínuos
x_edges = np.arange(x_min_aligned, x_max + grid_size, grid_size)
y_edges = np.arange(y_min_aligned, y_max + grid_size, grid_size)

# Criar células contínuas
grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Plot: Grelha contínua + pontos ortho
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

grid.boundary.plot(ax=ax, color='red', linewidth=0.3, alpha=0.8, label='Grelha (100m contínua)')
#gdf_ortho.plot(ax=ax, color='white', edgecolor='black', markersize=25, alpha=0.9, label='Pontos ORTHO (centros)')
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Grelha contínua centrada nos pontos ORTHO (100m)", fontsize=13)
ax.legend()
ax.set_axis_off()

plt.tight_layout()
plt.show()


# Pontos das órbitas ascendente e descendente

In [ ]:
# import pandas as pd
import geopandas as gpd
# import matplotlib.pyplot as plt
# from shapely.geometry import Point
import contextily as ctx

# # ==============================
# # 1. Ler CSVs ASC e DESC
# # ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

# ==============================
# 3. Criar GeoDataFrames e converter para Web Mercator
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=[Point(xy) for xy in zip(asc['easting'], asc['northing'])],
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=[Point(xy) for xy in zip(desc['easting'], desc['northing'])],
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Determinar limites comuns
# ==============================
x_min = min(gdf_asc.total_bounds[0], gdf_desc.total_bounds[0])
x_max = max(gdf_asc.total_bounds[2], gdf_desc.total_bounds[2])
y_min = min(gdf_asc.total_bounds[1], gdf_desc.total_bounds[1])
y_max = max(gdf_asc.total_bounds[3], gdf_desc.total_bounds[3])

# ==============================
# 5. Plot
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.7)
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_xlim([x_min, x_max])
axes[0].set_ylim([y_min, y_max])
axes[0].set_title("Ascending")
axes[0].set_axis_off()

gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.7)
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_xlim([x_min, x_max])
axes[1].set_ylim([y_min, y_max])
axes[1].set_title("Descending")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

# Pontos das órbitas ascendente e descendente + pontos ortho + grelha

## Mapas separados

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Caminhos dos datasets
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# ORTHO: dV e dH (atenção à troca de E/U nas pastas)
ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_dh_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

# ==============================
# 2. Ler datasets
# ==============================
asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_dv = pd.read_csv(ortho_dv_file)
ortho_dh = pd.read_csv(ortho_dh_file)

# O dataset "ortho_dv" é o principal para as coordenadas
ortho = ortho_dv.copy()

# ==============================
# 3. Definir área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 50  # metros

# ==============================
# 4. Filtrar área
# ==============================
asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 5. Criar GeoDataFrames
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=gpd.points_from_xy(asc['easting'], asc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=gpd.points_from_xy(desc['easting'], desc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 6. Criar grelha contínua centrada nos ORTHO
# ==============================
x_min = ortho['easting'].min() - grid_size / 2
x_max = ortho['easting'].max() + grid_size / 2
y_min = ortho['northing'].min() - grid_size / 2
y_max = ortho['northing'].max() + grid_size / 2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 7. Plot ASC e DESC com grelha e ORTHO
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# ASCENDING
grid.boundary.plot(ax=axes[0], color='red', linewidth=0.8, alpha=0.8)
gdf_ortho.plot(ax=axes[0], color='white', edgecolor='black', markersize=25, alpha=0.9, label='ORTHO (centro)')
gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.7, label='ASC')
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_title("Ascending (ASC) — grelha centrada nos ORTHO", fontsize=13)
axes[0].legend()
axes[0].set_axis_off()

# DESCENDING
grid.boundary.plot(ax=axes[1], color='red', linewidth=0.8, alpha=0.8)
gdf_ortho.plot(ax=axes[1], color='white', edgecolor='black', markersize=25, alpha=0.9, label='ORTHO (centro)')
gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.7, label='DESC')
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_title("Descending (DESC) — grelha centrada nos ORTHO", fontsize=13)
axes[1].legend()
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

## Mesmo mapa

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Definir parâmetros
# ==============================
grid_size = 100  # metros
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

# ==============================
# 2. Filtrar ASC/DESC e ORTHO na área
# ==============================
asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Criar GeoDataFrames
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=gpd.points_from_xy(asc['easting'], asc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=gpd.points_from_xy(desc['easting'], desc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Criar grelha contínua centrada nos pontos ORTHO
# ==============================
x_min = ortho['easting'].min() - grid_size / 2
x_max = ortho['easting'].max() + grid_size / 2
y_min = ortho['northing'].min() - grid_size / 2
y_max = ortho['northing'].max() + grid_size / 2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Plot
# ==============================
fig, ax = plt.subplots(figsize=(6,6))

# Grelha
grid.boundary.plot(ax=ax, color='red', linewidth=0.8, alpha=0.8, label='Grelha (100m contínua)')

# Pontos ORTHO
gdf_ortho.plot(ax=ax, color='white', edgecolor='black', markersize=25, alpha=0.9, label='ORTHO (centro)')

# Pontos ASC/DESC
gdf_asc.plot(ax=ax, color='blue', markersize=15, alpha=0.7, label='ASC')
gdf_desc.plot(ax=ax, color='yellow', markersize=15, alpha=0.7, label='DESC')

# Base de mapa
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

ax.set_title("Grelha contínua centrada nos ORTHO + pontos ASC/DESC", fontsize=14)
ax.legend()
ax.set_axis_off()
plt.tight_layout()
plt.show()

# Mapa com ID das células

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC e DESC
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

# ==============================
# 3. Definir datas comuns e interpolar
# ==============================
def melt_to_long(df):
    """Transforma colunas de datas em formato longo"""
    disp_cols = df.columns[24:]  # ajuste se necessário
    long_df = df.melt(id_vars=['easting','northing'], 
                      value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'date': common_dates,
            'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Definir grelha fixa
# ==============================
grid_size = 25  # metros

xmin = min(asc_interp['easting'].min(), desc_interp['easting'].min())
xmax = max(asc_interp['easting'].max(), desc_interp['easting'].max())
ymin = min(asc_interp['northing'].min(), desc_interp['northing'].min())
ymax = max(asc_interp['northing'].max(), desc_interp['northing'].max())

x_edges = np.arange(xmin, xmax + grid_size, grid_size)
y_edges = np.arange(ymin, ymax + grid_size, grid_size)

# ==============================
# 5. Função para pontos médios por célula
# ==============================
def grid_average(df, grid_size, x_edges, y_edges):
    df['cell_x'] = pd.cut(df['easting'], bins=x_edges, labels=False).astype('Int64')
    df['cell_y'] = pd.cut(df['northing'], bins=y_edges, labels=False).astype('Int64')
    
    valid = df.dropna(subset=['cell_x','cell_y']).copy()
    valid['x_center'] = x_edges[valid['cell_x'].to_numpy()] + grid_size/2
    valid['y_center'] = y_edges[valid['cell_y'].to_numpy()] + grid_size/2
    
    grouped = valid.groupby(['cell_x','cell_y','date']).agg(
        x_center=('x_center','first'),
        y_center=('y_center','first'),
        disp=('disp','mean')
    ).reset_index()
    
    # Adicionar cell_id
    grouped['cell_id'] = grouped['cell_x'].astype(str) + "_" + grouped['cell_y'].astype(str)
    
    return grouped

asc_cells = grid_average(asc_interp, grid_size, x_edges, y_edges)
desc_cells = grid_average(desc_interp, grid_size, x_edges, y_edges)

# ==============================
# 6. GeoDataFrames para plot
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc_cells,
    geometry=gpd.points_from_xy(asc_cells['x_center'], asc_cells['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc_cells,
    geometry=gpd.points_from_xy(desc_cells['x_center'], desc_cells['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Criar grelha como polígonos com IDs
# ==============================
grid_data = []
for ix, x0 in enumerate(x_edges[:-1]):
    for iy, y0 in enumerate(y_edges[:-1]):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x0, y0, x0+grid_size, y0+grid_size)
        })

grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 8. Plot com IDs
# ==============================
margin = grid_size * 0.05

fig, axes = plt.subplots(1, 2, figsize=(16,8))

# ASCENDING
grid.boundary.plot(ax=axes[0], color='red', linewidth=1, alpha=0.8)
gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.9)
for _, row in grid.iterrows():
    minx, miny, maxx, maxy = row['geometry'].bounds
    axes[0].text(maxx - margin, maxy - margin, row['cell_id'],
                 fontsize=6, ha='right', va='top', color='black',
                 bbox=dict(facecolor='white', alpha=0.7, pad=0.3))
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_title("Ascending")
axes[0].set_axis_off()

# DESCENDING
grid.boundary.plot(ax=axes[1], color='red', linewidth=1, alpha=0.8)
gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.9)
for _, row in grid.iterrows():
    minx, miny, maxx, maxy = row['geometry'].bounds
    axes[1].text(maxx - margin, maxy - margin, row['cell_id'],
                 fontsize=6, ha='right', va='top', color='black',
                 bbox=dict(facecolor='white', alpha=0.7, pad=0.3))
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_title("Descending")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC e DESC
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

# ==============================
# 3. Definir datas comuns e interpolar
# ==============================
def melt_to_long(df):
    """Transforma colunas de datas em formato longo"""
    disp_cols = df.columns[24:]  # ajuste se necessário
    long_df = df.melt(id_vars=['easting','northing'], 
                      value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'date': common_dates,
            'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Definir grelha fixa
# ==============================
grid_size = 100  # metros

xmin = min(asc_interp['easting'].min(), desc_interp['easting'].min())
xmax = max(asc_interp['easting'].max(), desc_interp['easting'].max())
ymin = min(asc_interp['northing'].min(), desc_interp['northing'].min())
ymax = max(asc_interp['northing'].max(), desc_interp['northing'].max())

x_edges = np.arange(xmin, xmax + grid_size, grid_size)
y_edges = np.arange(ymin, ymax + grid_size, grid_size)

# ==============================
# 5. Função para pontos médios por célula
# ==============================
def grid_average(df, grid_size, x_edges, y_edges):
    df['cell_x'] = pd.cut(df['easting'], bins=x_edges, labels=False).astype('Int64')
    df['cell_y'] = pd.cut(df['northing'], bins=y_edges, labels=False).astype('Int64')
    
    valid = df.dropna(subset=['cell_x','cell_y']).copy()
    valid['x_center'] = x_edges[valid['cell_x'].to_numpy()] + grid_size/2
    valid['y_center'] = y_edges[valid['cell_y'].to_numpy()] + grid_size/2
    
    grouped = valid.groupby(['cell_x','cell_y','date']).agg(
        x_center=('x_center','first'),
        y_center=('y_center','first'),
        disp=('disp','mean')
    ).reset_index()
    
    # Adicionar cell_id
    grouped['cell_id'] = grouped['cell_x'].astype(str) + "_" + grouped['cell_y'].astype(str)
    
    return grouped

asc_cells = grid_average(asc_interp, grid_size, x_edges, y_edges)
desc_cells = grid_average(desc_interp, grid_size, x_edges, y_edges)

# ==============================
# 6. GeoDataFrames para plot
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc_cells,
    geometry=gpd.points_from_xy(asc_cells['x_center'], asc_cells['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc_cells,
    geometry=gpd.points_from_xy(desc_cells['x_center'], desc_cells['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Criar grelha como polígonos com IDs
# ==============================
grid_data = []
for ix, x0 in enumerate(x_edges[:-1]):
    for iy, y0 in enumerate(y_edges[:-1]):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x0, y0, x0+grid_size, y0+grid_size)
        })

grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 8. Plot com IDs
# ==============================
margin = grid_size * 0.05

fig, axes = plt.subplots(1, 2, figsize=(16,8))

# ASCENDING
grid.boundary.plot(ax=axes[0], color='red', linewidth=1, alpha=0.8)
gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.9)
for _, row in grid.iterrows():
    minx, miny, maxx, maxy = row['geometry'].bounds
    axes[0].text(maxx - margin, maxy - margin, row['cell_id'],
                 fontsize=6, ha='right', va='top', color='black',
                 bbox=dict(facecolor='white', alpha=0.7, pad=0.3))
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_title("Ascending")
axes[0].set_axis_off()

# DESCENDING
grid.boundary.plot(ax=axes[1], color='red', linewidth=1, alpha=0.8)
gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.9)
for _, row in grid.iterrows():
    minx, miny, maxx, maxy = row['geometry'].bounds
    axes[1].text(maxx - margin, maxy - margin, row['cell_id'],
                 fontsize=6, ha='right', va='top', color='black',
                 bbox=dict(facecolor='white', alpha=0.7, pad=0.3))
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_title("Descending")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC e DESC
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Interpolação temporal ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar se necessário
    long_df = df.melt(id_vars=['easting','northing'], 
                      value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'date': common_dates,
            'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Criar grelha centrada nos pontos ORTHO
# ==============================
grid_size = 50  # metros

xmin = ortho['easting'].min() - grid_size/2
xmax = ortho['easting'].max() + grid_size/2
ymin = ortho['northing'].min() - grid_size/2
ymax = ortho['northing'].max() + grid_size/2

x_edges = np.arange(xmin, xmax + grid_size, grid_size)
y_edges = np.arange(ymin, ymax + grid_size, grid_size)

grid_data = []
for ix, x0 in enumerate(x_edges[:-1]):
    for iy, y0 in enumerate(y_edges[:-1]):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x0, y0, x0 + grid_size, y0 + grid_size)
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Função para média por célula
# ==============================
def grid_average(df, grid_size, x_edges, y_edges):
    df['cell_x'] = pd.cut(df['easting'], bins=x_edges, labels=False).astype('Int64')
    df['cell_y'] = pd.cut(df['northing'], bins=y_edges, labels=False).astype('Int64')
    
    valid = df.dropna(subset=['cell_x','cell_y']).copy()
    valid['x_center'] = x_edges[valid['cell_x'].to_numpy()] + grid_size/2
    valid['y_center'] = y_edges[valid['cell_y'].to_numpy()] + grid_size/2
    
    grouped = valid.groupby(['cell_x','cell_y','date']).agg(
        x_center=('x_center','first'),
        y_center=('y_center','first'),
        disp=('disp','mean')
    ).reset_index()
    
    grouped['cell_id'] = grouped['cell_x'].astype(str) + "_" + grouped['cell_y'].astype(str)
    return grouped

asc_cells = grid_average(asc_interp, grid_size, x_edges, y_edges)
desc_cells = grid_average(desc_interp, grid_size, x_edges, y_edges)

# ==============================
# 6. Converter para GeoDataFrames
# ==============================
def create_gdf(df):
    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['x_center'], df['y_center']),
        crs="EPSG:3035"
    ).to_crs(epsg=3857)

gdf_asc = create_gdf(asc_cells)
gdf_desc = create_gdf(desc_cells)
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Plot com IDs e alinhamento ORTHO
# ==============================
fig, ax = plt.subplots(figsize=(12,12))

# Grelha centrada nos ORTHO
grid.boundary.plot(ax=ax, color='red', linewidth=1, alpha=0.8, label=f'Grelha Base ({grid_size} m)')

# Pontos ORTHO (centro de referência)
gdf_ortho.plot(ax=ax, color='black', markersize=25, label='Pontos ORTHO')

# Pontos ASC/DESC (sobrepostos)
gdf_asc.plot(ax=ax, color='white', edgecolor='black', markersize=35, alpha=0.9, label='ASC')
gdf_desc.plot(ax=ax, color='yellow', markersize=25, alpha=0.8, label='DESC')

# IDs das células
for _, row in grid.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax.text(
        x_max - grid_size*0.05, y_max - grid_size*0.05, row['cell_id'],
        fontsize=6, ha='right', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=0.3)
    )

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Grelha centrada nos ORTHO com pontos ASC/DESC", fontsize=14)
ax.legend(loc='lower left', fontsize=9)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC e DESC
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Interpolação temporal ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar se necessário
    long_df = df.melt(id_vars=['easting','northing'], 
                      value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'date': common_dates,
            'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Criar grelha centrada nos pontos ORTHO
# ==============================
grid_size = 100  # metros

xmin = ortho['easting'].min() - grid_size/2
xmax = ortho['easting'].max() + grid_size/2
ymin = ortho['northing'].min() - grid_size/2
ymax = ortho['northing'].max() + grid_size/2

x_edges = np.arange(xmin, xmax + grid_size, grid_size)
y_edges = np.arange(ymin, ymax + grid_size, grid_size)

grid_data = []
for ix, x0 in enumerate(x_edges[:-1]):
    for iy, y0 in enumerate(y_edges[:-1]):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x0, y0, x0 + grid_size, y0 + grid_size)
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Função para média por célula
# ==============================
def grid_average(df, grid_size, x_edges, y_edges):
    df['cell_x'] = pd.cut(df['easting'], bins=x_edges, labels=False).astype('Int64')
    df['cell_y'] = pd.cut(df['northing'], bins=y_edges, labels=False).astype('Int64')
    
    valid = df.dropna(subset=['cell_x','cell_y']).copy()
    valid['x_center'] = x_edges[valid['cell_x'].to_numpy()] + grid_size/2
    valid['y_center'] = y_edges[valid['cell_y'].to_numpy()] + grid_size/2
    
    grouped = valid.groupby(['cell_x','cell_y','date']).agg(
        x_center=('x_center','first'),
        y_center=('y_center','first'),
        disp=('disp','mean')
    ).reset_index()
    
    grouped['cell_id'] = grouped['cell_x'].astype(str) + "_" + grouped['cell_y'].astype(str)
    return grouped

asc_cells = grid_average(asc_interp, grid_size, x_edges, y_edges)
desc_cells = grid_average(desc_interp, grid_size, x_edges, y_edges)

# ==============================
# 6. Converter para GeoDataFrames
# ==============================
def create_gdf(df):
    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['x_center'], df['y_center']),
        crs="EPSG:3035"
    ).to_crs(epsg=3857)

gdf_asc = create_gdf(asc_cells)
gdf_desc = create_gdf(desc_cells)
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Plot com IDs e alinhamento ORTHO
# ==============================
fig, ax = plt.subplots(figsize=(12,12))

# Grelha centrada nos ORTHO
grid.boundary.plot(ax=ax, color='red', linewidth=1, alpha=0.8, label=f'Grelha Base ({grid_size} m)')

# Pontos ORTHO (centro de referência)
gdf_ortho.plot(ax=ax, color='black', markersize=25, label='Pontos ORTHO')

# Pontos ASC/DESC (sobrepostos)
gdf_asc.plot(ax=ax, color='white', edgecolor='black', markersize=35, alpha=0.9, label='ASC')
gdf_desc.plot(ax=ax, color='yellow', markersize=25, alpha=0.8, label='DESC')

# IDs das células
for _, row in grid.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax.text(
        x_max - grid_size*0.05, y_max - grid_size*0.05, row['cell_id'],
        fontsize=6, ha='right', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=0.3)
    )

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Grelha centrada nos ORTHO com pontos ASC/DESC", fontsize=14)
ax.legend(loc='lower left', fontsize=9)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filtrar_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filtrar_area(asc)
desc = filtrar_area(desc)
ortho_v = filtrar_area(ortho_v)
ortho_h = filtrar_area(ortho_h)

# ==============================
# 3. Criar grelha base (100 m)
# ==============================
grid_size = 100
x_edges = np.arange(asc['easting'].min(), asc['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc['northing'].min(), asc['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

# Atribuir células ASC
asc['cell_x'] = pd.cut(asc['easting'], bins=x_edges_shifted, labels=False)
asc['cell_y'] = pd.cut(asc['northing'], bins=y_edges_shifted, labels=False)

# Calcular centroide de cada célula com base nos pontos ASC
agg = asc.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 4. Criar GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

points = gpd.GeoDataFrame(
    agg,
    geometry=gpd.points_from_xy(agg['x_center'], agg['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 5. Atribuir células a ORTHO
# ==============================
for ortho_df in [ortho_v, ortho_h]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

# ==============================
# 6. Células selecionadas
# ==============================
selected_ids = [
    "2_5","3_5","4_5","5_5","6_5","7_5",
    "2_4","3_4","4_4","5_4","6_4","7_4",
    "2_6","3_6","4_6","5_6","6_6","7_6"
]

agg_sel = agg[agg['cell_id'].isin(selected_ids)]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = points[points['cell_id'].isin(selected_ids)]

ortho_v_sel = ortho_v[ortho_v['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h[ortho_h['cell_id'].isin(selected_ids)]

# Combinar ORTHO
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing'])
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Mapa Final
# ==============================
fig, ax = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax, color='lightgray', linewidth=0.5, label='Grelha Base (100 m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax, color='red', linewidth=1.5, label='Células Selecionadas')

# Pontos ASC/DESC
points_sel.plot(ax=ax, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Pontos ORTHO
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Basemap e título
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax.set_axis_off()
ax.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()
